In [ ]:
#!/usr/bin/env python3
"""
Yahoo Finance News Scraper - Updated for 2025
Fixed selectors based on current Yahoo Finance HTML structure
"""

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import (
    TimeoutException, 
    NoSuchElementException, 
    StaleElementReferenceException,
    WebDriverException,
    ElementClickInterceptedException,
    ElementNotInteractableException
)

from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import random
from datetime import datetime
import re
import json
from urllib.parse import urljoin, urlparse
import logging
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

class YahooFinanceNewsScraper:
    def __init__(self, headless=False):
        self.driver = None
        self.base_url = "https://finance.yahoo.com"
        self.news_url = "https://finance.yahoo.com/news/"
        self.headless = headless
        self.all_articles = []
        
        # Setup logging
        logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
        self.logger = logging.getLogger(__name__)
        
        # Updated categories for 2025
        self.categories = {
            'Latest': '/news/',
            'Stock Market': '/topic/stock-market-news',
            'Tech': '/topic/technology',
            'Earnings': '/topic/earnings',
            'Crypto': '/topic/crypto-currencies',
            'Economy': '/topic/economic-news',
        }
    
    def setup_driver(self):
        """Setup Chrome driver with optimized options"""
        chrome_options = Options()
        
        if self.headless:
            chrome_options.add_argument('--headless=new')
        
        chrome_options.add_argument('--disable-gpu')
        chrome_options.add_argument('--no-sandbox')
        chrome_options.add_argument('--disable-dev-shm-usage')
        chrome_options.add_argument('--disable-blink-features=AutomationControlled')
        chrome_options.add_experimental_option('excludeSwitches', ['enable-automation'])
        chrome_options.add_experimental_option('useAutomationExtension', False)
        chrome_options.add_argument('--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
        
        # Performance optimizations
        chrome_options.add_argument('--disable-plugins')
        chrome_options.add_argument('--disable-extensions')
        
        try:
            self.driver = webdriver.Chrome(
                service=Service(ChromeDriverManager().install()), 
                options=chrome_options
            )
            self.driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
            self.driver.implicitly_wait(10)
            self.driver.set_window_size(1920, 1080)
            self.logger.info("Chrome driver initialized successfully")
        except Exception as e:
            self.logger.error(f"Failed to initialize driver: {e}")
            raise
    
    def handle_consent_modal(self):
        """Handle cookie consent modal if it appears"""
        try:
            # Updated selectors for 2025 consent modal
            consent_selectors = [
                'button[name="agree"]',
                'button:contains("Accept all")',
                '.consent-overlay button',
                '[data-testid="consent-accept"]',
                'button.btn-primary'
            ]
            
            for selector in consent_selectors:
                try:
                    accept_button = WebDriverWait(self.driver, 3).until(
                        EC.element_to_be_clickable((By.CSS_SELECTOR, selector))
                    )
                    accept_button.click()
                    self.logger.info("Cookie consent accepted")
                    time.sleep(2)
                    return
                except TimeoutException:
                    continue
                    
        except Exception as e:
            self.logger.debug(f"No consent modal found or error handling: {e}")
    
    def random_delay(self, min_delay=1, max_delay=3):
        """Add random delay to avoid being detected as bot"""
        time.sleep(random.uniform(min_delay, max_delay))
    
    def scroll_to_load_more(self, max_scrolls=8):
        """Scroll down to load more articles with better loading strategy"""
        current_height = 0
        
        for i in range(max_scrolls):
            # Get current page height
            new_height = self.driver.execute_script("return document.body.scrollHeight")
            
            # Scroll to bottom
            self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            self.random_delay(2, 4)
            
            # Try to find and click load more button
            try:
                load_more_selectors = [
                    "button[data-testid='load-more-stories']",
                    "button:contains('Load More')",
                    ".load-more-button",
                    "button.btn-load-more",
                    "button[aria-label*='Load more']",
                    ".btn-load-more",
                    "[data-testid*='load-more']"
                ]
                
                clicked = False
                for selector in load_more_selectors:
                    try:
                        load_more_button = self.driver.find_element(By.CSS_SELECTOR, selector)
                        if load_more_button.is_displayed() and load_more_button.is_enabled():
                            self.driver.execute_script("arguments[0].click();", load_more_button)
                            self.logger.info(f"Clicked load more button (scroll {i+1})")
                            self.random_delay(3, 6)
                            clicked = True
                            break
                    except:
                        continue
                
                if clicked:
                    # Wait for new content to load
                    self.random_delay(3, 5)
                    
            except:
                pass
            
            # Check if page height changed (new content loaded)
            final_height = self.driver.execute_script("return document.body.scrollHeight")
            if final_height == current_height:
                # No new content loaded, scroll a bit more and try again
                self.driver.execute_script("window.scrollBy(0, 1000);")
                self.random_delay(2, 3)
            
            current_height = final_height
        
        # Scroll back to top gradually to avoid missing content
        self.driver.execute_script("window.scrollTo(0, 0);")
        self.random_delay(2, 3)
    
    def extract_article_data(self, article_element):
        """Extract data from a single article element with improved parsing for 2025"""
        try:
            article_data = {}
            
            # Get the full text content
            article_text = article_element.text.strip()
            if not article_text or len(article_text) < 20:
                return None
            
            # Split into lines and clean
            lines = [line.strip() for line in article_text.split('\n') if line.strip()]
            if len(lines) < 1:
                return None
            
            # Extract title (first substantial line)
            title = None
            summary_start_idx = 1
            
            for i, line in enumerate(lines):
                if len(line) > 15 and not any(x in line.lower() for x in ['ago', '•', 'advertisement']):
                    title = line
                    summary_start_idx = i + 1
                    break
            
            if not title or len(title) > 300:
                return None
                
            article_data['title'] = title
            
            # Look for URL - try multiple approaches
            url_found = False
            try:
                # Try to find any link within this element
                link_elements = article_element.find_elements(By.CSS_SELECTOR, 'a[href]')
                for link in link_elements:
                    href = link.get_attribute('href')
                    if href and ('yahoo.com' in href and ('/news/' in href or '/finance/' in href)):
                        article_data['url'] = href
                        url_found = True
                        break
            except:
                pass
            
            if not url_found:
                article_data['url'] = "N/A"
            
            # Parse remaining lines for summary, source, and time
            summary_lines = []
            source_found = False
            time_found = False
            
            for line in lines[summary_start_idx:]:
                # Check for source and time pattern (contains •)
                if '•' in line:
                    parts = line.split('•')
                    if len(parts) >= 2:
                        potential_source = parts[0].strip()
                        potential_time = parts[1].strip()
                        
                        # Validate source (shouldn't be too long or contain numbers)
                        if (len(potential_source) < 50 and 
                            not potential_source.isdigit() and
                            not any(x in potential_source.lower() for x in ['ago', 'hour', 'min', 'day'])):
                            article_data['source'] = potential_source
                            source_found = True
                        
                        # Validate time (should contain time indicators)
                        if any(x in potential_time.lower() for x in ['ago', 'minute', 'hour', 'day']):
                            article_data['published_time'] = potential_time
                            time_found = True
                            
                            # Try to convert to datetime
                            try:
                                article_data['published_datetime'] = self.parse_relative_time(potential_time)
                            except:
                                article_data['published_datetime'] = "N/A"
                    continue
                
                # Check for standalone time (line with 'ago')
                elif any(x in line.lower() for x in ['ago', 'minutes ago', 'hours ago', 'days ago']) and not time_found:
                    article_data['published_time'] = line.strip()
                    try:
                        article_data['published_datetime'] = self.parse_relative_time(line.strip())
                    except:
                        article_data['published_datetime'] = "N/A"
                    time_found = True
                    continue
                
                # This looks like summary content
                elif (len(line) > 25 and 
                      not any(x in line.lower() for x in ['advertisement', 'sponsored', 'click here']) and
                      not source_found):
                    summary_lines.append(line)
            
            # Build summary
            if summary_lines:
                # Take first 3 lines of summary, up to 400 characters
                full_summary = ' '.join(summary_lines[:3])
                if len(full_summary) > 400:
                    full_summary = full_summary[:400] + "..."
                article_data['summary'] = full_summary
            else:
                article_data['summary'] = "N/A"
            
            # Set defaults for missing fields
            if not source_found:
                article_data['source'] = "Yahoo Finance"
            if not time_found:
                article_data['published_time'] = "N/A"
                article_data['published_datetime'] = "N/A"
            
            # Try to extract image URL
            try:
                img_element = article_element.find_element(By.CSS_SELECTOR, 'img[src]')
                img_src = img_element.get_attribute('src')
                if img_src and ('yahoo' in img_src or 'yimg' in img_src):
                    article_data['image_url'] = img_src
                else:
                    article_data['image_url'] = "N/A"
            except:
                article_data['image_url'] = "N/A"
            
            article_data['scraped_at'] = datetime.now().isoformat()
            
            return article_data
                
        except Exception as e:
            self.logger.error(f"Error extracting article data: {e}")
            return None
    
    def parse_relative_time(self, time_str):
        """Convert relative time string to datetime"""
        try:
            import re
            from datetime import timedelta
            
            time_str = time_str.lower().strip()
            now = datetime.now()
            
            # Extract number and unit
            match = re.search(r'(\d+)\s*(minute|hour|day|week|month)s?\s*ago', time_str)
            if match:
                num = int(match.group(1))
                unit = match.group(2)
                
                if unit == 'minute':
                    return (now - timedelta(minutes=num)).isoformat()
                elif unit == 'hour':
                    return (now - timedelta(hours=num)).isoformat()
                elif unit == 'day':
                    return (now - timedelta(days=num)).isoformat()
                elif unit == 'week':
                    return (now - timedelta(weeks=num)).isoformat()
                elif unit == 'month':
                    return (now - timedelta(days=num*30)).isoformat()
            
            return "N/A"
        except:
            return "N/A"
    
    def scrape_category(self, category_name, category_url, max_articles=150):
        """Scrape articles from a specific category with updated approach for 2025"""
        self.logger.info(f"Scraping category: {category_name}")
        
        full_url = urljoin(self.base_url, category_url)
        
        try:
            self.driver.get(full_url)
            self.random_delay(3, 5)
            
            # Handle consent modal
            self.handle_consent_modal()
            
            # Wait for page to load completely
            try:
                WebDriverWait(self.driver, 15).until(
                    lambda driver: driver.execute_script("return document.readyState") == "complete"
                )
            except TimeoutException:
                self.logger.warning("Page did not fully load")
            
            # Wait a bit more for dynamic content
            self.random_delay(3, 5)
            
            # Scroll to load more content aggressively
            self.scroll_to_load_more(max_scrolls=8)
            
            # Try multiple strategies to find articles
            articles = []
            
            # Strategy 1: Look for specific article containers
            article_container_selectors = [
                '[data-testid*="stream"] > div',
                '[data-testid*="content"] > div', 
                '[data-module="ContentStream"] > div',
                '.stream-items > li',
                'article',
                '.js-content-viewer > div',
            ]
            
            for selector in article_container_selectors:
                try:
                    found_articles = self.driver.find_elements(By.CSS_SELECTOR, selector)
                    if found_articles and len(found_articles) > len(articles):
                        articles = found_articles
                        self.logger.info(f"Found {len(articles)} articles using selector: {selector}")
                        break
                except Exception as e:
                    self.logger.debug(f"Selector {selector} failed: {e}")
                    continue
            
            # Strategy 2: Look for divs containing news-related content
            if len(articles) < 10:  # If we didn't find many articles, try broader search
                try:
                    # Get the main content area
                    main_selectors = [
                        'div[data-testid="content-area"]',
                        '#main-content',
                        '.content-area',
                        'main',
                        '.content-wrapper',
                        'body'
                    ]
                    
                    main_content = None
                    for main_sel in main_selectors:
                        try:
                            main_content = self.driver.find_element(By.CSS_SELECTOR, main_sel)
                            break
                        except:
                            continue
                    
                    if main_content:
                        # Find all divs that might contain articles
                        all_divs = main_content.find_elements(By.CSS_SELECTOR, 'div')
                        potential_articles = []
                        
                        for div in all_divs:
                            try:
                                text = div.text.strip()
                                # Check if this looks like an article
                                if (100 < len(text) < 2000 and  # Reasonable length
                                    ('ago' in text.lower() or '•' in text) and  # Has time indicator
                                    not any(x in text.lower() for x in ['advertisement', 'sponsored', 'cookie']) and
                                    text.count('\n') > 0):  # Multi-line content
                                    potential_articles.append(div)
                            except:
                                continue
                        
                        if len(potential_articles) > len(articles):
                            articles = potential_articles
                            self.logger.info(f"Found {len(articles)} articles using broad div search")
                
                except Exception as e:
                    self.logger.debug(f"Broad search failed: {e}")
            
            # Strategy 3: Last resort - find any div with article-like text patterns
            if len(articles) < 5:
                try:
                    all_elements = self.driver.find_elements(By.CSS_SELECTOR, '*')
                    article_elements = []
                    
                    for element in all_elements:
                        try:
                            text = element.text.strip()
                            # Very broad check for article-like content
                            if (200 < len(text) < 1500 and
                                any(x in text for x in ['•', 'ago', 'Yahoo', 'Finance']) and
                                text.count('\n') >= 2):
                                article_elements.append(element)
                                if len(article_elements) >= 50:  # Limit to avoid processing too many
                                    break
                        except:
                            continue
                    
                    if len(article_elements) > len(articles):
                        articles = article_elements
                        self.logger.info(f"Found {len(articles)} articles using last resort search")
                        
                except Exception as e:
                    self.logger.debug(f"Last resort search failed: {e}")
            
            if not articles:
                self.logger.warning(f"No articles found for category: {category_name}")
                return []
            
            self.logger.info(f"Processing {len(articles)} potential articles from {category_name}")
            
            category_articles = []
            articles_to_process = min(len(articles), max_articles)
            successful_extractions = 0
            
            # Process articles with better filtering
            for i in tqdm(range(len(articles)), desc=f"Scraping {category_name}"):
                if successful_extractions >= max_articles:
                    break
                    
                try:
                    article = articles[i]
                    article_data = self.extract_article_data(article)
                    
                    if (article_data and 
                        article_data.get('title') and 
                        len(article_data['title']) > 15 and
                        article_data['title'] != 'N/A'):
                        
                        # Check for duplicates
                        is_duplicate = False
                        for existing in category_articles:
                            if existing['title'] == article_data['title']:
                                is_duplicate = True
                                break
                        
                        if not is_duplicate:
                            article_data['category'] = category_name
                            article_data['category_url'] = category_url
                            category_articles.append(article_data)
                            successful_extractions += 1
                            self.logger.debug(f"Extracted: {article_data['title'][:50]}...")
                        
                except Exception as e:
                    self.logger.debug(f"Error processing article {i}: {e}")
                    continue
                
                self.random_delay(0.1, 0.2)
            
            self.logger.info(f"Successfully scraped {successful_extractions} articles from {category_name}")
            return category_articles
            
        except Exception as e:
            self.logger.error(f"Error scraping category {category_name}: {e}")
            return []
    
    def scrape_all_categories(self, max_articles_per_category=150):
        """Scrape all categories with increased article limits"""
        self.logger.info("Starting to scrape all Yahoo Finance news categories")
        
        all_articles = []
        
        for category_name, category_url in self.categories.items():
            try:
                category_articles = self.scrape_category(category_name, category_url, max_articles_per_category)
                all_articles.extend(category_articles)
                
                # Delay between categories
                self.random_delay(5, 8)
                
            except Exception as e:
                self.logger.error(f"Failed to scrape category {category_name}: {e}")
                continue
        
        self.all_articles = all_articles
        self.logger.info(f"Total articles scraped: {len(all_articles)}")
        return all_articles
    
    def save_to_csv(self, filename=None):
        """Save scraped articles to CSV"""
        if not self.all_articles:
            self.logger.warning("No articles to save")
            return
        
        if filename is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"yahoo_finance_news_{timestamp}.csv"
        
        df = pd.DataFrame(self.all_articles)
        df.to_csv(filename, index=False, encoding='utf-8')
        self.logger.info(f"Saved {len(self.all_articles)} articles to {filename}")
        
        # Display stats
        print(f"\nScraping Summary:")
        print(f"Total articles: {len(self.all_articles)}")
        if len(self.all_articles) > 0:
            print(f"Categories covered: {df['category'].unique()}")
            print(f"Articles per category:")
            print(df['category'].value_counts())
        
        return filename
    
    def save_to_json(self, filename=None):
        """Save scraped articles to JSON"""
        if not self.all_articles:
            self.logger.warning("No articles to save")
            return
        
        if filename is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"yahoo_finance_news_{timestamp}.json"
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(self.all_articles, f, indent=2, ensure_ascii=False)
        
        self.logger.info(f"Saved {len(self.all_articles)} articles to {filename}")
        return filename
    
    def debug_page_structure(self, url=None):
        """Debug method to understand page structure"""
        if url is None:
            url = self.news_url
            
        self.logger.info(f"Debugging page structure for: {url}")
        
        try:
            self.driver.get(url)
            self.random_delay(5, 7)
            
            # Print page title
            print(f"Page title: {self.driver.title}")
            
            # Print first 1000 characters of body text
            body_text = self.driver.find_element(By.TAG_NAME, 'body').text
            print(f"Body text (first 1000 chars):\n{body_text[:1000]}")
            
            # Find all divs with text content
            divs_with_text = []
            all_divs = self.driver.find_elements(By.CSS_SELECTOR, 'div')
            
            for i, div in enumerate(all_divs[:50]):  # Check first 50 divs
                try:
                    text = div.text.strip()
                    if 50 < len(text) < 500:  # Reasonable article length
                        divs_with_text.append((i, text[:100]))
                except:
                    continue
            
            print(f"\nFound {len(divs_with_text)} divs with reasonable text content:")
            for i, text in divs_with_text[:10]:  # Show first 10
                print(f"Div {i}: {text}...")
                
        except Exception as e:
            self.logger.error(f"Error debugging page structure: {e}")
    
    def close(self):
        """Close the driver"""
        if self.driver:
            self.driver.quit()
            self.logger.info("Driver closed successfully")

def main():
    """Main execution function with debug option"""
    scraper = YahooFinanceNewsScraper(headless=False)  # Keep non-headless for debugging
    
    try:
        scraper.setup_driver()
        
        # Uncomment to debug page structure first
        # scraper.debug_page_structure()
        # return
        
        articles = scraper.scrape_all_categories(max_articles_per_category=50)
        
        if articles:
            csv_file = scraper.save_to_csv()
            json_file = scraper.save_to_json()
            
            print(f"\nFiles saved:")
            print(f"CSV: {csv_file}")
            print(f"JSON: {json_file}")
            
            # Show sample data
            if len(articles) > 0:
                print(f"\nSample article:")
                print(f"Title: {articles[0]['title']}")
                print(f"Summary: {articles[0]['summary'][:100]}...")
                print(f"Source: {articles[0]['source']}")
        else:
            print("No articles were scraped.")
            print("Running debug mode to understand page structure...")
            scraper.debug_page_structure()
    
    except Exception as e:
        print(f"An error occurred: {e}")
        # Run debug mode if main scraping fails
        try:
            scraper.debug_page_structure()
        except:
            pass
    
    finally:
        scraper.close()

if __name__ == "__main__":
    main()

2025-06-14 12:17:10,872 - INFO - ====== WebDriver manager ======
2025-06-14 12:17:14,328 - INFO - Get LATEST chromedriver version for google-chrome
2025-06-14 12:17:14,387 - INFO - Get LATEST chromedriver version for google-chrome
2025-06-14 12:17:14,441 - INFO - Get LATEST chromedriver version for google-chrome
2025-06-14 12:17:14,574 - INFO - WebDriver version 137.0.7151.70 selected
2025-06-14 12:17:14,578 - INFO - Modern chrome version https://storage.googleapis.com/chrome-for-testing-public/137.0.7151.70/mac-arm64/chromedriver-mac-arm64.zip
2025-06-14 12:17:14,578 - INFO - About to download new driver from https://storage.googleapis.com/chrome-for-testing-public/137.0.7151.70/mac-arm64/chromedriver-mac-arm64.zip
2025-06-14 12:17:14,622 - INFO - Driver downloading response is 200
2025-06-14 12:17:14,965 - INFO - Get LATEST chromedriver version for google-chrome
2025-06-14 12:17:15,106 - INFO - Driver has been saved in cache [/Users/aadityajoshi/.wdm/drivers/chromedriver/mac64/137.0.

In [ ]:
from sqlalchemy import create_engine
df = pd.read_csv('yahoo_finance_news_20250609_014201.csv')

In [ ]:
df = df[['title','summary','category']]

In [ ]:
df

In [ ]:
df.dropna(inplace=True)

In [ ]:
df

In [ ]:
import numpy as np
from transformers import pipeline
%matplotlib inline

# Pipelines
classifier = pipeline("text-classification", model="Sharpaxis/Finance_DistilBERT_sentiment", top_k=None)
pipe = pipeline("text-classification", model="Sharpaxis/News_classification_distilbert")

def sentiment_predictor(row):
    text = row['title']
    out = classifier(text)[0]
    # Sentiment analysis scores
    scores = [sample['score'] for sample in out]
    labels = [sample['label'] for sample in out]
    label_map = {'LABEL_0': "Negative", 'LABEL_1': "Neutral", 'LABEL_2': "Positive"}
    sentiments = [label_map[label] for label in labels]
    
    sentiment = sentiments[np.argmax(scores)]
    return sentiment
def fin_news_predictor(row):
    text = row['title']
    type_news = pipe(text)[0]
    if type_news['label'] == 'LABEL_1':
        return "FIN_NEWS"
    else:
        return "NON-FIN-NEWS"


In [ ]:
df['Sentiment'] = df.apply(sentiment_predictor,axis=1)
df['Fin_news'] = df.apply(fin_news_predictor,axis=1)

In [ ]:
df